# NYC Watershed Satellite Embeddings Analysis

This notebook analyzes satellite embedding vectors from Google Earth Engine to detect changes in the NYC watershed region between 2017 and 2024.

## Overview

- **Data Source**: Google Satellite Embedding V1 Annual dataset
- **Region**: 50km buffer around NYC watershed center point (-74.9710, 42.2554)
- **Time Periods**: 2017 and 2024

## Maps & Layers

1. **Clustering Analysis**: K-means clustering on embedding bands (6, 10, 20 clusters) comparing 2017 vs 2024
   - Shows spatial patterns in embedding space at different granularities
   
2. **Euclidean Distance**: Per-pixel embedding vector distance between 2024 and 2017
   - White = high change, Black = low change
   
3. **Top 3 Difference Bands**: Visualizes the three embedding bands with greatest change
   - Individual band differences (2024 - 2017)
   - RGB composite of top 3 bands for both years side-by-side

In [12]:
import ee
import geemap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


ee.Authenticate()
ee.Initialize(project="gsapp-map")

dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")


center = [-74.9710, 42.2554]
point = ee.Geometry.Point(-74.9710, 42.2554)


region = point.buffer(50000)

image2017 = (
    dataset.filterDate("2017-01-01", "2018-01-01")
    .filterBounds(region)
    .mosaic()
    .clip(region)  # Add this to actually clip pixels to the region
    .reproject(crs="EPSG:2263", scale=100)
)
image2024 = (
    dataset.filterDate("2024-01-01", "2025-01-01")
    .filterBounds(region)
    .mosaic()
    .clip(region)  # Add this to actually clip pixels to the region
    .reproject(crs="EPSG:2263", scale=100)
)

# Visualize three axes of the embedding space as an RGB.
visParams = {min: -0.3, max: 0.3, "bands": ["A31", "A42", "A03"]}

Map = geemap.Map(center=center, zoom=7)
Map.add_basemap("TopPlusOpen.Grey")

Map.addLayer(image2017, visParams, "2017 embeddings")
Map.addLayer(image2024, visParams, "2024 embeddings")

Map.centerObject(point, zoom=7)
Map.setOptions("SATELLITE")
Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [13]:
def get_cmap_palette(n_clusters, cmap_name="viridis"):
    cmap = plt.get_cmap(cmap_name)
    colors = [mcolors.to_hex(cmap(i / (n_clusters - 1))) for i in range(n_clusters)]
    return colors


# Sample points for training
n_samples = 1000

training2017 = image2017.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)
training2024 = image2024.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)
combined_training = training2017.merge(training2024)
bands = list(
    set(image2017.bandNames().getInfo()) & set(image2024.bandNames().getInfo())
)

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

clusters = [6, 10, 20]

for c in clusters:
    clusterer = ee.Clusterer.wekaKMeans(c).train(
        features=combined_training, inputProperties=bands
    )

    palette = get_cmap_palette(c, cmap_name="tab20")

    Map.addLayer(
        image2017.cluster(clusterer).visualize(min=0, max=c - 1, palette=palette),
        {},
        f"2017 : {c} Clusters",
    )
    Map.addLayer(
        image2024.cluster(clusterer).visualize(min=0, max=c - 1, palette=palette),
        {},
        f"2024 : {c} Clusters",
    )

Map.centerObject(point, zoom=9)
Map.setOptions("SATELLITE")
Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [14]:
# Compute per-pixel Euclidean distance between embedding vectors
diff = image2024.select(bands).subtract(image2017.select(bands))
sq = diff.pow(2)
sum_sq = sq.reduce(ee.Reducer.sum())
euclidean_dist = sum_sq.sqrt().rename("euclidean_dist")

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

# Visualize the Euclidean distance (difference magnitude)
# white = high difference, black = low difference
Map.addLayer(
    euclidean_dist,
    {"min": 0, "max": 0.5, "palette": ["black", "white"]},
    "Embedding Difference (Euclidean)",
)

Map.centerObject(point, zoom=9)
Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…